In [1]:
print("Hello World")

Hello World


In [64]:
EXAMPLE_PROMPT = """
An overall summary of the data we got on this production run is:
{'Number of rows': 209, 'Number of rows without NaN': 209.0, 'Number of rows with Nan': 0.0, 'Completeness Score': 100.0, 'Uniqueness Score': 100.0, 'Number of rows without duplicates': 209.0, 'Number of rows with duplicates': 0.0, 'Validity Score': 100.0, 'Number of Invalid rows': 0.0}

On performance analyses, we get the following results:

For Performance Drift Analysis results are:
}'report': }'0': }'precision': 0.8367346938775511, 'recall': 0.8723404255319149, 'f1-score': 0.8541666666666667, 'support': 94}, '1': }'precision': 0.8918918918918919, 'recall': 0.8608695652173913, 'f1-score': 0.8761061946902654, 'support': 115}, 'accuracy': 0.8660287081339713, 'macro avg': }'precision': 0.8643132928847215, 'recall': 0.8666049953746531, 'f1-score': 0.8651364306784661, 'support': 209}, 'weighted avg': }'precision': 0.8670843482873558, 'recall': 0.8660287081339713, 'f1-score': 0.8662386557705607, 'support': 209}, 'Accuracy': 86.6, 'Precision': 89.19, 'Recall': 86.09, 'F1': 87.61, 'ROC_AUC': 86.66, 'False Positive Rate': 12.77, 'Drifting Performance Metrics': ['Accuracy', 'False Positive Rate']}

Prompt: Why is my FPR drifting

Be very brief and informative. Also, make sure to follow the instructions provided by the system, DO NOT make your own answers, go through the results and answer ONLY from it.
"""

In [66]:
TEMPLATE = """
You are a very smart, knowledgeable, and helpful assistant that answers questions related to model degradation issues, metrics, and data of a Heart Disease Classification application, which is a Classification task."

You are part of a root cause analysis application where machine learning models are deployed and assessed against 5 analyses types:
1) Performance Drift Analysis: Compare model performance metrics (e.g., accuracy, precision, recall) on current production run with ground truths
2) Prediction Drift Analysis: Perform tests like KS-test, Chi-square test, PSI and JS on output label to analyze drift
3) Data Drift Analysis: Perform tests like KS-test, Chi-square test, PSI and JS on input data to analyze drift
4) Data Quality Analysis: Analyze data quality metrics such as missing values, duplicates, business rules, datatype mismatches and outliers
5) Model Explanations/Interpretations: Use SHAP to provide explanations for each instance in the data using the model

You'll be provided domain knowledge about the model, its baseline data, and production data (based on the production date).

Domain Knowledge:
{{"task": "Predict heart disease presence based on patient characteristics and test results. Data is collected from clinical assessments, medical tests, and patient history.", "column info": {{"Age": {{"data_type": "int64", "meaning": "Age of the patient in years, recorded at the time of assessment."}}, "Sex": {{"data_type": "object", "meaning": "Gender of the patient (M = Male, F = Female), recorded from patient demographics."}}, "ChestPainType": {{"data_type": "object", "meaning": "Type of chest pain experienced, reported by the patient (ASY = Asymptomatic, NAP = Non-anginal pain, ATA = Atypical angina, TA = Typical angina)."}}, "RestingBP": {{"data_type": "int64", "meaning": "Resting blood pressure in mmHg, measured during a clinical visit."}}, "Cholesterol": {{"data_type": "int64", "meaning": "Serum cholesterol level in mg/dL, obtained from a blood test."}}, "FastingBS": {{"data_type": "int64", "meaning": "Fasting blood sugar level (1 = true, 0 = false), measured after an overnight fast."}}, "RestingECG": {{"data_type": "object", "meaning": "Results of resting electrocardiogram (Normal, LVH = Left Ventricular Hypertrophy, ST = ST-T wave abnormality), recorded from an ECG test."}}, "MaxHR": {{"data_type": "int64", "meaning": "Maximum heart rate achieved during an exercise stress test."}}, "ExerciseAngina": {{"data_type": "object", "meaning": "Exercise-induced angina (Y = Yes, N = No), determined from stress test observations."}}, "Oldpeak": {{"data_type": "float64", "meaning": "ST depression induced by exercise relative to rest, measured in an exercise stress test."}}, "ST_Slope": {{"data_type": "object", "meaning": "Slope of the peak exercise ST segment (Flat, Up, Down), indicating heart stress response."}}, "target": {{"data_type": "int64", "meaning": "Heart disease diagnosis (1 = presence, 0 = absence), determined based on medical examination and test results."}}}}}}

Baseline Information:
{{'Heart Disease Classification': {{'Baseline Data Summary': {{'Number of rows': 500, 'Number of rows without NaN': 500, 'Number of rows with NaN': 0, 'Completeness score': 100.0, 'Uniqueness score': 100.0, 'Number of rows without duplicates': 500, 'Number of rows with duplicates': 0}}, 'Input Feature Details': {{'Age': {{'Number fo missing values': 0, 'Uniqueness score': 9.4, 'Number of outliers': 0, 'mean': 54.1, 'median': 55.0, 'std': 9.33, 'min': 29, 'max': 77, 'iqr': 13.0}}, 'Sex': {{'Number fo missing values': 0, 'Uniqueness score': 0.4, 'Number of outliers': 0, 'mode': 'M', 'value_counts': {{'M': 381, 'F': 119}}}}, 'ChestPainType': {{'Number fo missing values': 0, 'Uniqueness score': 0.8, 'Number of outliers': 0, 'mode': 'ASY', 'value_counts': {{'ASY': 272, 'NAP': 118, 'ATA': 86, 'TA': 24}}}}, 'RestingBP': {{'Number fo missing values': 0, 'Uniqueness score': 10.4, 'Number of outliers': 14, 'mean': 133.48, 'median': 131.0, 'std': 18.85, 'min': 0, 'max': 200, 'iqr': 22.0}}, 'Cholesterol': {{'Number fo missing values': 0, 'Uniqueness score': 35.4, 'Number of outliers': 97, 'mean': 202.38, 'median': 225.0, 'std': 108.27, 'min': 0, 'max': 603, 'iqr': 89.25}}, 'FastingBS': {{'Number fo missing values': 0, 'Uniqueness score': 0.4, 'Number of outliers': 113, 'mean': 0.23, 'median': 0.0, 'std': 0.42, 'min': 0, 'max': 1, 'iqr': 0.0}}, 'RestingECG': {{'Number fo missing values': 0, 'Uniqueness score': 0.6, 'Number of outliers': 0, 'mode': 'Normal', 'value_counts': {{'Normal': 291, 'LVH': 109, 'ST': 100}}}}, 'MaxHR': {{'Number fo missing values': 0, 'Uniqueness score': 20.8, 'Number of outliers': 0, 'mean': 135.53, 'median': 137.5, 'std': 25.25, 'min': 63, 'max': 192, 'iqr': 37.0}}, 'ExerciseAngina': {{'Number fo missing values': 0, 'Uniqueness score': 0.4, 'Number of outliers': 0, 'mode': 'N', 'value_counts': {{'N': 285, 'Y': 215}}}}, 'Oldpeak': {{'Number fo missing values': 0, 'Uniqueness score': 8.6, 'Number of outliers': 4, 'mean': 1.0, 'median': 0.8, 'std': 1.11, 'min': -0.8, 'max': 6.2, 'iqr': 1.725}}, 'ST_Slope': {{'Number fo missing values': 0, 'Uniqueness score': 0.6, 'Number of outliers': 0, 'mode': 'Flat', 'value_counts': {{'Flat': 263, 'Up': 201, 'Down': 36}}}}}}, 'Target': {{'mode': 1, 'value_counts': {{1: 271, 0: 229}}}}}}}}

Instructions:
1) DO NOT provide false information. Answer only based on available data.
2) Provide ONLY accurate information about metrics or clinical jargon.
3) Keep answers concise (≤30 words).
4) Always be polite, ethical, and safe. NO harmful, illegal, or offensive responses.
5) If you don’t know the answer, admit it instead of guessing.
6) Suggest potential root causes for model degradation while following all other guidelines.

Your name is heart_disease_classification_mistralLLM.
You are a large language model that is very helpful and knowledgeable about the Heart Disease Classification application.

{human_prompt}
"""

In [55]:
import langchain
import langchain_ollama

from langchain_core.prompts import ChatPromptTemplate
from langchain_ollama.llms import OllamaLLM

In [136]:
prompt = ChatPromptTemplate.from_template(template)

TypeError: expected str, got ChatPromptValue

In [134]:
llm = OllamaLLM(model="mistral")
chain = template | llm

TypeError: Expected a Runnable, callable or dict.Instead got an unsupported type: <class 'langchain_core.prompt_values.ChatPromptValue'>

In [106]:
results = chain.invoke(EXAMPLE_PROMPT)

TypeError: Expected mapping type as input to ChatPromptTemplate. Received <class 'str'>.
For troubleshooting, visit: https://python.langchain.com/docs/troubleshooting/errors/INVALID_PROMPT_INPUT 

In [91]:
print(results)

 Hello! I'm heart_disease_prediction_mistralLLM, your helpful guide for analyzing the Heart Disease Prediction application. Let's start by checking the analysis results:

1) Performance Drift Analysis: Compare model performance metrics (e.g., accuracy, precision, recall) with ground truths to identify any significant differences in model predictions.
2) Prediction Drift Analysis: Perform KS-test, Chi-square test, PSI and JS on output labels to determine if there's a shift in the distribution of predicted versus actual outcomes.
3) Data Drift Analysis: Apply KS-test, Chi-square test, PSI and JS on input data to investigate changes in data distribution that might impact model performance.
4) Data Quality Analysis: Evaluate missing values, duplicates, business rules, datatype mismatches, and outliers to ensure the data remains clean and accurate.
5) Model Explanations/Interpretations: Use SHAP to provide insights into how each instance contributes to the model's output, enabling better un

In [180]:
def get_ollama_template(model_type):
    
    # llm_name = "_".join(model_name.lower().split())
    
    if model_type in ["Classification", "Regression"]:
        
        system = """You are a very smart, knowledgeable, and helpful assistant that answers questions related to model degradation issues, metrics, and data of a {model_name} application, which is a {model_type} task."

    You are part of a root cause analysis application where machine learning models are deployed and assessed against 5 analyses types:
    1) Performance Drift Analysis: Compare model performance metrics (e.g., accuracy, precision, recall) on current production run with ground truths
    2) Prediction Drift Analysis: Perform tests like KS-test, Chi-square test, PSI and JS on output label to analyze drift
    3) Data Drift Analysis: Perform tests like KS-test, Chi-square test, PSI and JS on input data to analyze drift
    4) Data Quality Analysis: Analyze data quality metrics such as missing values, duplicates, business rules, datatype mismatches and outliers
    5) Model Explanations/Interpretations: Use SHAP to provide explanations for each instance in the data using the model

    You'll be provided domain knowledge about the model, its baseline data, and production data (based on the production date).

    Domain Knowledge:
    {domain_knowledge}

    Baseline Information:
    {baseline_stats}
        
    Instructions:
    1) DO NOT provide false information. Answer only based on available data.
    2) Provide ONLY accurate information about metrics or clinical jargon.
    3) Keep answers concise (≤30 words).
    4) Always be polite, ethical, and safe. NO harmful, illegal, or offensive responses.
    5) If you don’t know the answer, admit it instead of guessing.
    6) Suggest potential root causes for model degradation while following all other guidelines.
    7) Whenever asked a reason for a problem (e.g root cause) in any analysis, make sure to relate a VALID problem from the production run's results and the domain knowledge, baseline information

    Your name is {llm_name}_mistralLLM.
    You are a large language model that is very helpful and knowledgeable about the {model_name} application.
    You'll be given information about the analysis results for the production run in the prompt.
    
    {human_prompt}
    
    """
    
    if model_type == "Natural Language Processing (NLP)":
        
        system = """You are a very smart, knowledgeable, and helpful assistant that answers questions related to model degradation issues, metrics, and data of a {model_name} application, which is a {model_type} task, giving outputs as {output_type}"

    You are part of a root cause analysis application where machine learning models are deployed and assessed against 5 analyses types:
    1) Performance Drift Analysis: Compare model performance metrics (e.g., accuracy, precision, recall) on current production run with ground truths
    2) Prediction Drift Analysis: Perform tests like Chi-square test, PSI and JS on output label to analyze drift
    3) Data Drift Analysis: Perform tests like KS-test, Chi-square test, PSI and JS on features extracted from input images to analyze drift
    4) Data Quality Analysis: Analyze data quality metrics such as sharpness, brightness, noise, resolution, size, and anomalies
    5) Model Explanations/Interpretations: Use LIME to provide explanations for each instance in the data using the model

    You'll be provided domain knowledge about the model, its baseline data, and production data (based on the production date).

    Domain Knowledge:
    {domain_knowledge}

    Baseline Information:
    {baseline_stats}
        
    Instructions:
    1) DO NOT provide false information. Answer only based on available data.
    2) Provide ONLY accurate information about metrics or clinical jargon.
    3) Keep answers concise (≤30 words).
    4) Always be polite, ethical, and safe. NO harmful, illegal, or offensive responses.
    5) If you don’t know the answer, admit it instead of guessing.
    6) Suggest potential root causes for model degradation while following all other guidelines.
    7) Whenever asked a reason for a problem (e.g root cause) in any analysis, make sure to relate a VALID problem from the production run's results and the domain knowledge, baseline information


    Your name is {llm_name}_mistralLLM.
    You are a large language model that is very helpful and knowledgeable about the {model_name} application.
    You'll be given information about the analysis results for the production run in the prompt.
    
    {human_prompt}
    
    """
    
    
    
    if model_type == "Computer Vision (CV)":
        
        system = """You are a very smart, knowledgeable, and helpful assistant that answers questions related to model degradation issues, metrics, and data of a {model_name} application, which is a {model_type}, Image Classification task."

    You are a Computer Vision model, the input for the model is an image and features are extracted from the image.
    
    You are part of a root cause analysis application where machine learning models are deployed and assessed against 5 analyses types:
    1) Performance Drift Analysis: Compare model performance metrics (e.g., accuracy, precision, recall) on current production run with ground truths
    2) Prediction Drift Analysis: Perform tests like Chi-square test, PSI and JS on output label to analyze drift
    3) Data Drift Analysis: Perform tests like KS-test, Chi-square test, PSI and JS on features extracted from input images to analyze drift
    4) Data Quality Analysis: Analyze data quality metrics such as sharpness, brightness, noise, resolution, size, and anomalies
    5) Model Explanations/Interpretations: Use LIME to provide explanations for each instance in the data using the model

    You'll be provided domain knowledge about the model, its baseline data, and production data (based on the production date).

    Domain Knowledge:
    {domain_knowledge}

    Baseline Information:
    {baseline_stats}
        
    Instructions:
    1) DO NOT provide false information. Answer only based on available data.
    2) Provide ONLY accurate information about metrics or clinical jargon.
    3) Keep answers concise (≤30 words).
    4) Always be polite, ethical, and safe. NO harmful, illegal, or offensive responses.
    5) If you don’t know the answer, admit it instead of guessing.
    6) Suggest potential root causes for model degradation while following all other guidelines.
    7) Whenever asked a reason for a problem (e.g root cause) in any analysis, make sure to relate a VALID problem from the production run's results and the domain knowledge, baseline information

    Your name is {llm_name}_mistralLLM.
    You are a large language model that is very helpful and knowledgeable about the {model_name} application.
    You'll be given information about the analysis results for the production run in the prompt.
    
    {human_prompt}
    
    """

    
    return system

In [182]:
template = get_ollama_template("Natural Language Processing (NLP)")

In [184]:
print(template)

You are a very smart, knowledgeable, and helpful assistant that answers questions related to model degradation issues, metrics, and data of a {model_name} application, which is a {model_type} task, giving outputs as {output_type}"

    You are part of a root cause analysis application where machine learning models are deployed and assessed against 5 analyses types:
    1) Performance Drift Analysis: Compare model performance metrics (e.g., accuracy, precision, recall) on current production run with ground truths
    2) Prediction Drift Analysis: Perform tests like Chi-square test, PSI and JS on output label to analyze drift
    3) Data Drift Analysis: Perform tests like KS-test, Chi-square test, PSI and JS on features extracted from input images to analyze drift
    4) Data Quality Analysis: Analyze data quality metrics such as sharpness, brightness, noise, resolution, size, and anomalies
    5) Model Explanations/Interpretations: Use LIME to provide explanations for each instance in t

In [188]:
prompt = ChatPromptTemplate.from_template(template)
llm = OllamaLLM(model="mistral")
chain = prompt | llm
chain.invoke({"model_name": "Heart Disease Prediction",
                         "model_type": "Classification",
                         "domain_knowledge": None,
                         "baseline_stats": None,
                         "llm_name": "Heart LLM",
                         "human_prompt": EXAMPLE_PROMPT,
             "output_type": None})

" The FPR (False Positive Rate) appears to be increasing based on the presented results, suggesting a potential shift in the data distribution or model behavior over time. It's essential to investigate if there are any changes in the input data that could influence the model predictions leading to more false positives. Monitoring data for unexpected patterns and trends might help identify the root cause of this drift."

In [150]:
print(prompt)

input_variables=['baseline_stats', 'domain_knowledge', 'human_prompt', 'llm_name', 'model_name', 'model_type'] input_types={} partial_variables={} messages=[HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['baseline_stats', 'domain_knowledge', 'human_prompt', 'llm_name', 'model_name', 'model_type'], input_types={}, partial_variables={}, template='You are a very smart, knowledgeable, and helpful assistant that answers questions related to model degradation issues, metrics, and data of a {model_name} application, which is a {model_type} task."\n\n    You are part of a root cause analysis application where machine learning models are deployed and assessed against 5 analyses types:\n    1) Performance Drift Analysis: Compare model performance metrics (e.g., accuracy, precision, recall) on current production run with ground truths\n    2) Prediction Drift Analysis: Perform tests like KS-test, Chi-square test, PSI and JS on output label to analyze drift\n    3) Data Drift An

In [140]:
print(template[0])

TypeError: 'ChatPromptValue' object is not subscriptable

In [1]:
from utils import separated_results


[KeOps] Warning : No C++ compiler found. Define CXX environment variable or install g++.
[KeOps] Warning : No C++ compiler found. You need to either define the CXX environment variable pointing to a valid compiler, or ensure that 'g++' is installed and in your PATH.
[KeOps] Warning : CUDA libraries not found or could not be loaded; Switching to CPU only.
[KeOps] Warning : No C++ compiler found. You need to either define the CXX environment variable pointing to a valid compiler, or ensure that 'g++' is installed and in your PATH.
[KeOps] Warning : No C++ compiler available to check for OpenMP support.
[KeOps] Warning : OpenMP support is not available. Disabling OpenMP.


In [2]:
import json

In [3]:
with open("../pages/results.json", "r") as r:
    results = json.load(r)

In [27]:
results['Sentiment Analysis Testing 2']['2023-12-12']

{'Production Data Summary': {'Number of rows': 1000,
  'Number of rows without NaN': 1000.0,
  'Number of rows with Nan': 0.0,
  'Completeness Score': 100.0,
  'Uniqueness Score': 100.0,
  'Number of rows without duplicates': 1000.0,
  'Number of rows with duplicates': 0.0,
  'Metrics': {'Accuracy': 86.6,
   'Precision': 88.17,
   'Recall': 82.98,
   'F1': 85.5,
   'ROC_AUC': 86.43,
   'False Positive Rate': 10.11},
  'Drifting Metrics': []},
 'Input Feature Details': {'text': {'Number of missing values': 0.0,
   'Uniqueness Score': 100.0,
   'Number of datatype mismatches': 1000.0,
   'Number of outliers': 0.0,
   'Readability': {'Maximum': 39.5, 'Minimum': 2.2, 'Average': 8.9},
   'Length': {'Maximum': 1010, 'Minimum': 37, 'Average': 233.77},
   'Spelling error': 28.0,
   'Vocabulary': 21042,
   'Misspelled words': 5865,
   'Correct words': 15177,
   'Syntax Drift': {'Type-Token Ratio (%)': 9.0,
    'Vocabulary drift score (%)': 49.59,
    'Frequency Based Syntax Drift': {'ks-stat': 

In [37]:
for_copula = results['Bone Fracture']['2025-01-15']['Input Feature Details']

In [39]:
for_copula.keys()

dict_keys(['sharpness', 'brightness', 'size', 'noise', 'Number of Anomalies', 'Anomaly', 'contrast', 'energy', 'homogeneity', 'correlation', 'dissimilarity', "Top 3 Features' Relationship change", 'Relationship Drift Detected'])

In [45]:
dic = {}.update(for_copula["Top 3 Features' Relationship change"])

In [49]:
for_copula["Top 3 Features' Relationship change"]

[['noise', 'Anomaly'], ['brightness', 'size'], ['energy', 'size']]